# Coffee Standard J25 — AF2 luminance isolation
Fresh 50-epoch AF2LUMDIRECT seed-42 screen. The AF2 cue is computed from Rec.709 luminance and shared across RGB. Existing matched D0DIRECT and AF2DIRECT reports are read-only references. Test is never extracted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import csv, importlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
ARM='AF2LUMDIRECT'; BRANCH='codex/af2-luminance-isolation'
REPO=Path('/content/coffee-bean-detection'); WORK=Path('/content')
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','gdown'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Colab')
from coffee_detector.drive_project import resolve_drive_project_root
from coffee_detector.analysis.coffee_standard_j25_thesis_provenance import audit_j25_thesis_provenance
from coffee_detector.data.prepare_coffee_standard_j25_source_split import prepare_j25_source_split
PROJECT=resolve_drive_project_root()
REFERENCE=PROJECT/'experiments/coffee-standard-j25-af2-direct-v2'
for name in ('D0DIRECT','AF2DIRECT'):
    report=REFERENCE/'val_reports'/f'{name}_seed42_result.json'
    if not report.is_file(): raise FileNotFoundError(f'Reference belum tersedia: {report}')
    payload=json.loads(report.read_text())
    if payload.get('protocol')!='coffee-standard-j25-train-siblings-af2-direct-seed42-v2' or payload.get('test_images_accessed') is not False: raise RuntimeError(f'Reference tidak valid: {report}')
ARCHIVE=WORK/'data_aug_11.zip'
if not ARCHIVE.is_file(): subprocess.run([sys.executable,'-m','gdown','https://drive.google.com/uc?id=1AofT7VbiNFM8ul-0vyCAKj7Rp4j5OX0f','-O',str(ARCHIVE)],check=True)
PROVENANCE=WORK/'coffee_standard_j25_thesis_provenance.json'
provenance=audit_j25_thesis_provenance(ARCHIVE,PROVENANCE)
if not provenance['decision'].startswith('PASS'): raise RuntimeError(f'Provenance gagal: {provenance["decision"]}')
DATA=WORK/'coffee-standard-j25-train-siblings-v2'
if DATA.exists(): shutil.rmtree(DATA)
contract=prepare_j25_source_split(ARCHIVE,DATA,seed=42,retain_train_siblings=True)
CONTRACT=DATA/'coffee_standard_j25_train_siblings_summary.json'
from ultralytics import YOLO
_=YOLO('yolo26n.pt'); PRETRAINED=REPO/'yolo26n.pt'
OUT=PROJECT/'experiments/coffee-standard-j25-af2-luminance-v1'; OUT.mkdir(parents=True,exist_ok=True)
print('ARM:',ARM,'GPU:',torch.cuda.get_device_name(0),'DATA:',contract['images'],'OUT:',OUT)


In [ ]:
LOG=OUT/f'{ARM}_seed42_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_coffee_standard_j25_af2_luminance','--data-root',str(DATA),'--development-contract',str(CONTRACT),'--provenance-summary',str(PROVENANCE),'--pretrained-checkpoint',str(PRETRAINED),'--output-root',str(OUT),'--seed','42','--device','0','--authorize-training']
print('START/RESUME:',ARM,'| log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
last=-1
while process.poll() is None:
    results=OUT/ARM/f'{ARM}_seed42'/'results.csv'
    epochs=sum(1 for _ in csv.DictReader(results.open(encoding='utf-8'))) if results.is_file() else 0
    if epochs!=last: print(f'{ARM}: {epochs}/50 epoch tercatat',flush=True); last=epochs
    time.sleep(120)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'{ARM} gagal: {process.returncode}')
RESULT=OUT/'val_reports'/f'{ARM}_seed42_result.json'
print(json.dumps(json.loads(RESULT.read_text()),indent=2,ensure_ascii=False))
print('last.pt tersimpan di Drive setiap epoch. Test tidak diekstrak.')


In [ ]:
from coffee_detector.experiments.run_coffee_standard_j25_af2_luminance import build_decision
DECISION=OUT/'af2_luminance_seed42_decision.json'
result=build_decision(REFERENCE,OUT,DECISION)
print('VALUES:',result['values'])
print('LUMINANCE minus RGB:',result['luminance_minus_rgb'])
print('LUMINANCE minus D0:',result['luminance_minus_native'])
print('INTERPRETATION:',result['interpretation'])
print('NEXT:',result['next'],'| TEST:',result['test_opened'])
